In [1]:
%matplotlib inline


Neural Networks

使用torch.nn包来构建神经网络。

上一讲已经讲过了``autograd``，``nn``包依赖``autograd``包来定义模型并求导。
一个``nn.Module``包含各个层和一个``forward(input)``方法，该方法返回``output``。



例如：

![](https://pytorch.org/tutorials/_images/mnist.png)

它是一个简单的前馈神经网络，它接受一个输入，然后一层接着一层地传递，最后输出计算的结果。

神经网络的典型训练过程如下：

1. 定义包含一些可学习的参数(或者叫权重)神经网络模型； 
2. 在数据集上迭代； 
3. 通过神经网络处理输入； 
4. 计算损失(输出结果和正确值的差值大小)；
5. 将梯度反向传播回网络的参数； 
6. 更新网络的参数，主要使用如下简单的更新原则： 
``weight = weight - learning_rate * gradient``

  

定义网络

开始定义一个网络：



In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        # nn.Conv2d 是 PyTorch 中用于构建二维卷积层的类，主要用于处理二维数据（比如图片：高度 × 宽度 × 通道数），核心功能是通过卷积核（kernel）对输入数据做卷积运算，提取数据的局部特征（比如图片的边缘、纹理、形状等）。
        #nn.Conv2d(in_channels, out_channels, kernel_size) 是二维卷积层的核心定义方式，三个参数分别对应输入通道数、输出通道数（卷积核数）、卷积核尺寸；
        # 1 input image channel, 6 output channels, 5x5 square convolution
        # 卷积核的大小，这里 5 等价于 (5,5)，表示每个卷积核是 5×5 的正方形矩阵。
        # kernel
        # 后一层卷积的 in_channels 必须等于前一层的 out_channels
        # 卷积核个数 = 输出通道数
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        # an affine operation: y = Wx + b
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    #流程：卷积→激活→池化 → 卷积→激活→池化 → 展平 → 全连接→激活 → 全连接→激活 → 最终全连接输出；
    def forward(self, x):
        #ReLU 激活函数，作用是引入非线性（把所有负数变成 0，保留正数），让模型能学习复杂特征；
        # Max pooling over a (2, 2) window
        #F.max_pool2d(..., (2, 2))：对 ReLU 输出做 2×2 最大池化，核心作用是 “下采样”—— 把特征图的尺寸缩小一半（比如 24×24 → 12×12），减少计算量，同时保留关键特征。
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)  # 2 等价于 (2, 2)
        # view 是 PyTorch 中调整张量形状的方法，变为 (batch_size, 16 * 5 * 5)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        #[1:]：切片操作，取从第 1 个维度开始的所有维度，目的是排除第 0 维的批次维度（batch_size）
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features


net = Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


在模型中必须要定义 ``forward`` 函数，``backward``
函数（用来计算梯度）会被``autograd``自动创建。
可以在 ``forward`` 函数中使用任何针对 Tensor 的操作。

 ``net.parameters()``返回可被学习的参数（权重）列表和值



In [10]:
#net.parameters() 会返回模型中所有需要梯度更新的参数（权重 weight + 偏置 bias），转成列表后，每个元素对应一层的权重或偏置。
params = list(net.parameters())
# print(params)
# 输出10 = 5*2 ，2为权重和偏置
print(len(params))
#torch.Size([6, 1, 5, 5])，含义：
# 第一个维度 6：输出通道数（卷积核个数）；
# 第二个维度 1：输入通道数；
# 第三 / 四个维度 5,5：卷积核的尺寸（5×5）；
# 最外层的 6 个大块：对应维度 0（6 个卷积核）；
# 每个大块里只有 1 个中块：对应维度 1（1 个输入通道）；
# 每个中块里是 5×5 的小数组：对应维度 2 和 3（卷积核的高和宽）；
print(params[0].size())  # conv1's .weight

[Parameter containing:
tensor([[[[ 0.1041, -0.0179, -0.1455,  0.1578, -0.1794],
          [-0.1354, -0.1923,  0.1113, -0.0866,  0.1029],
          [-0.1314,  0.1021, -0.1576,  0.0309, -0.1982],
          [ 0.0808, -0.1867, -0.1597,  0.1904, -0.0068],
          [ 0.1235, -0.0541,  0.1459,  0.1121, -0.1546]]],


        [[[ 0.1507, -0.0270, -0.1884,  0.0724,  0.1482],
          [-0.1405, -0.1809, -0.1795, -0.0780,  0.0852],
          [-0.1970, -0.1828,  0.1820,  0.1115, -0.0697],
          [ 0.0753, -0.1889, -0.1189,  0.1281, -0.0003],
          [-0.0577,  0.0300,  0.1272,  0.0039,  0.1934]]],


        [[[ 0.1400,  0.0309, -0.0347, -0.0508, -0.1869],
          [-0.1911,  0.1848,  0.0562,  0.0897, -0.0527],
          [-0.1833, -0.0161, -0.1298, -0.1114,  0.0486],
          [ 0.1892, -0.0198, -0.0833,  0.1542, -0.1142],
          [ 0.1981,  0.1800, -0.0202,  0.1025, -0.1257]]],


        [[[-0.0381,  0.0727, -0.1127,  0.1393,  0.0212],
          [-0.1442, -0.0062, -0.0480, -0.0303,  0.079

测试随机输入32×32。
注：这个网络（LeNet）期望的输入大小是32×32，如果使用MNIST数据集来训练这个网络，请把图片大小重新调整到32×32。



In [11]:
# 形状符合 “批次 × 通道 × 高 × 宽” 的 4 维要求，且通道数和 conv1 匹配；图片尺寸要足够大，能支撑卷积 + 池化的计算（比如 32×32 适配 5×5 卷积核）。
# 输入 (1, 1, 32, 32)
# → conv1(1→6, 5×5核) → (1, 6, 28, 28)  （32-5+1=28）
# → ReLU → 形状不变
# → max_pool2d(2×2) → (1, 6, 14, 14)   （28/2=14）
# → conv2(6→16,5×5核) → (1, 16, 10, 10)（14-5+1=10）
# → ReLU → 形状不变
# → max_pool2d(2×2) → (1, 16, 5, 5)    （10/2=5）
# → 展平 → (1, 16×5×5=400)
# → fc1(400→120) → (1, 120)
# → fc2(120→84) → (1, 84)
# → fc3(84→10) → (1, 10) （最终输出，对应0-9的预测分数）
input = torch.randn(1, 1, 32, 32)
print(input)
out = net(input)
print(out)

tensor([[[[ 1.0167e+00, -6.4179e-01,  1.5238e-01,  ...,  4.5564e-01,
           -1.7615e+00, -2.2255e-01],
          [ 1.7079e-02, -5.0088e-01,  7.6238e-01,  ...,  4.1459e-01,
           -2.3641e-01,  1.4066e+00],
          [-2.2733e+00, -1.2481e+00, -8.6179e-01,  ...,  1.1636e-01,
            1.0153e+00, -1.3025e+00],
          ...,
          [ 1.4799e+00, -4.5122e-01,  1.9806e+00,  ..., -7.2384e-01,
           -3.4070e-01, -1.4820e+00],
          [-1.0317e+00,  4.4794e-01,  1.9728e-03,  ..., -3.4464e-01,
            2.2180e-01, -4.1070e-01],
          [-1.8331e+00,  2.9012e-01, -4.7090e-01,  ..., -1.1696e+00,
           -8.2265e-01, -8.4156e-01]]]])
tensor([[ 0.1381, -0.0228, -0.0200,  0.0030,  0.0346,  0.0654,  0.0158, -0.0840,
          0.0954,  0.0016]], grad_fn=<AddmmBackward0>)


将所有参数的梯度缓存清零，然后进行随机梯度的的反向传播：



In [5]:
net.zero_grad()
out.backward(torch.randn(1, 10))

<div class="alert alert-info"><h4>Note</h4><p>``torch.nn`` 只支持小批量输入。整个 ``torch.nn``
包都只支持小批量样本，而不支持单个样本。

    例如，``nn.Conv2d`` 接受一个4维的张量，
    ``每一维分别是sSamples * nChannels * Height * Width（样本数*通道数*高*宽）``。

    如果你有单个样本，只需使用 ``input.unsqueeze(0)`` 来添加其它的维数</p></div>

在继续之前，我们回顾一下到目前为止用到的类。

**回顾:**
  -  ``torch.Tensor``：一个用过自动调用 ``backward()``实现支持自动梯度计算的 *多维数组* ，
      并且保存关于这个向量的*梯度* w.r.t.
  -  ``nn.Module``：神经网络模块。封装参数、移动到GPU上运行、导出、加载等。
  -  ``nn.Parameter``：一种变量，当把它赋值给一个``Module``时，被 *自动* 地注册为一个参数。
  -  ``autograd.Function``：实现一个自动求导操作的前向和反向定义，每个变量操作至少创建一个函数节点，每一个``Tensor``的操作都回创建一个接到创建``Tensor``和 *编码其历史* 的函数的``Function``节点。

**重点如下：**
  -  定义一个网络
  -  处理输入，调用backword

**还剩：**
  -  计算损失
  -  更新网络权重

损失函数
一个损失函数接受一对 (output, target) 作为输入，计算一个值来估计网络的输出和目标值相差多少。

***译者注：output为网络的输出，target为实际值***

nn包中有很多不同的[损失函数](https://pytorch.org/docs/nn.html#loss-functions)。
``nn.MSELoss``是一个比较简单的损失函数，它计算输出和目标间的**均方误差**，
例如：



In [12]:
output = net(input)
print(output.shape)  # torch.Size([1, 10])
target = torch.randn(10)  # 随机值作为样例
print(target.size()) # torch.Size([10])
target = target.view(1, -1)  # 使target和output的shape相同 torch.Size([1, 10])
print(target.size())
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

torch.Size([1, 10])
torch.Size([10])
torch.Size([1, 10])
tensor(2.0384, grad_fn=<MseLossBackward0>)


现在，如果在反向过程中跟随``loss`` ， 使用它的
``.grad_fn`` 属性，将看到如下所示的计算图。

::

    input -> conv2d -> relu -> maxpool2d -> conv2d -> relu -> maxpool2d
          -> view -> linear -> relu -> linear -> relu -> linear
          -> MSELoss
          -> loss

所以，当我们调用 ``loss.backward()``时,整张计算图都会
根据loss进行微分，而且图中所有设置为``requires_grad=True``的张量
将会拥有一个随着梯度累积的``.grad`` 张量。

为了说明，让我们向后退几步:



In [13]:
print(loss.grad_fn)  # MSELoss
# next_functions 本身是一个列表，列表中的每个元素是一个二元组 (grad_fn, index)：
# 第一个元素 grad_fn：当前运算的输入张量对应的梯度函数（反向传播的上游节点）。
# 第二个元素 index：当前运算的输入张量在其生成函数的输出中的索引（多输出函数时有用，单输出时通常为 0）。
# 通过嵌套 next_functions[0][0]，可以沿着计算图反向遍历整个梯度传播链路，从损失函数一直追溯到输入层的运算。
print(loss.grad_fn.next_functions[0][0])  # Linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0])  # ReLU

反向传播
调用loss.backward()获得反向传播的误差。

但是在调用前需要清除已存在的梯度，否则梯度将被累加到已存在的梯度。

现在，我们将调用loss.backward()，并查看conv1层的偏差（bias）项在反向传播前后的梯度。




In [8]:
net.zero_grad()     # 清除梯度

print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)

loss.backward()

print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)

conv1.bias.grad before backward
None
conv1.bias.grad after backward
tensor([ 0.0028, -0.0080, -0.0063, -0.0003, -0.0011,  0.0132])


如何使用损失函数

**稍后阅读：**

  `nn`包，包含了各种用来构成深度神经网络构建块的模块和损失函数，完整的文档请查看[here](https://pytorch.org/docs/nn)。

**剩下的最后一件事:**

  - 新网络的权重

更新权重
在实践中最简单的权重更新规则是随机梯度下降（SGD）：

     ``weight = weight - learning_rate * gradient``

我们可以使用简单的Python代码实现这个规则：

```python

learning_rate = 0.01
for f in net.parameters():
    f.data.sub_(f.grad.data * learning_rate)
```
但是当使用神经网络是想要使用各种不同的更新规则时，比如SGD、Nesterov-SGD、Adam、RMSPROP等，PyTorch中构建了一个包``torch.optim``实现了所有的这些规则。
使用它们非常简单：


In [9]:
import torch.optim as optim

# create your optimizer
optimizer = optim.SGD(net.parameters(), lr=0.01)

# in your training loop:
optimizer.zero_grad()   # zero the gradient buffers
output = net(input)
loss = criterion(output, target)
loss.backward()
optimizer.step()    # Does the update

.. 注意::
    
      观察如何使用``optimizer.zero_grad()``手动将梯度缓冲区设置为零。
      这是因为梯度是按Backprop部分中的说明累积的。

